### Task 1: Build the Core LCEL Chain with a ChatPromptTemplate

Create a new Jupyter notebook called smart_qa_bot.ipynb. Import ChatOpenAI and build a ChatPromptTemplate with a system message that defines your bot as a helpful assistant, and a human message placeholder for {question}. Use the LCEL pipe operator (|) to chain: prompt | llm | StrOutputParser(). Invoke the chain with a sample question and print the response. Verify the chain runs without errors before moving to the next task.

In [6]:
from langchain_core.prompts import ChatPromptTemplate

from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser

# get ANTHROPIC_API_KEY env var from .env file
load_dotenv()


def core_lcel():
    model = init_chat_model(
            model="claude-sonnet-4-5-20250929",
            model_provider="anthropic",
            temperature=0.7,
            streaming=False,
            max_retries=3,
        )
    
    prompt = ChatPromptTemplate.from_messages([
                ("system", "You are a helpful assistant."),
                ("human", "{question}")
            ])

    parser = StrOutputParser()

    chain = prompt | model | parser

    response = chain.invoke({"question": "What is the capital of Italy?"})

    print(f"Response: {response}")

core_lcel()

Response: The capital of Italy is Rome.


### Task 2: Add Structured Output with a Pydantic Model

Define a Pydantic BaseModel called QAResponse with three fields: answer (str), confidence (float between 0 and 1), and sources (List[str]). Replace StrOutputParser() in your chain with llm.with_structured_output(QAResponse). Invoke the chain with a question and print the structured result, accessing each field individually (e.g., result.answer, result.confidence). Confirm the output is a typed Python object, not a plain string.

In [15]:
from pydantic import BaseModel, Field
from typing import List

load_dotenv()

class QAResponse(BaseModel):
    answer: str = Field(description="ai answer for the questions")
    confidence: float = Field(description="confidence level between 0 and 1", ge=0.0, le=1.0)
    sources: List[str] = Field(description="sources", default_factory=list)
    

def add_structured_output_with_pydeantic_model():
    model = init_chat_model(
            model="claude-sonnet-4-5-20250929",
            model_provider="anthropic",
            temperature=0.7,
            streaming=False,
            max_retries=3,
        ).with_structured_output(QAResponse)
    

    prompt = ChatPromptTemplate.from_messages([
                    ("system", "You are a helpful assistant."),
                    ("human", "{question}")
                ])
    
    chain = prompt | model

    result = chain.invoke({"question": "How old is the Earth?"})

    print(type(result))
    print(f"""
Answer: {result.answer}, 
Confidence: {result.confidence}, 
Sources: {result.sources}""")


add_structured_output_with_pydeantic_model()


<class '__main__.QAResponse'>

Answer: The Earth is approximately 4.54 billion years old. This age has been determined through radiometric dating of rocks and meteorites, particularly through the analysis of the oldest known terrestrial rocks and moon rocks, as well as meteorites that formed at the same time as our solar system. Scientists use various dating methods, with uranium-lead dating of zircon crystals being one of the most reliable techniques. The age of 4.54 billion years (with an uncertainty of about 50 million years) represents the time since Earth formed from the solar nebula along with the rest of the solar system., 
Confidence: 0.99, 
Sources: ['Scientific consensus based on radiometric dating methods', 'Geological and planetary science research']


### Task 3: Configure Multi-Provider Fallback with OpenAI and Anthropic

Instantiate both ChatOpenAI (gpt-4o-mini) and ChatAnthopic (claude-3-haiku-20240307). Use the .with_fallbacks() method to create a resilient primary_llm that falls back to Anthropic if OpenAI fails. Build a new chain: prompt | primary_llm | StrOutputParser(). Test the fallback by first running a normal invocation, then simulate a failure by passing an invalid OpenAI API key to a separate ChatOpenAI instance and confirm the chain automatically uses Anthropic. Print which model produced each response.

In [ ]:
# from langchain_anthropic import ChatAnthropic
#from langchain_openai import ChatOpenAI
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv

load_dotenv()

# llm_openai = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)
# llm_openai.with_fallbacks()
# llm_anthropic = ChatAnthropic(model="claude-sonnet-4-5-20250929", temperature=0)


primary_llm = init_chat_model(
    model="gpt-4o-mini",
    model_provider="openai",
    temperature=0.3,
    streaming=False,
    max_retries=3,
)


fallback_llm = init_chat_model(
    model="claude-sonnet-4-5-20250929",
    model_provider="anthropic",
    temperature=0.7,
    streaming=False,
    max_retries=3,
)

model = primary_llm.with_fallbacks([fallback_llm])

parser = StrOutputParser()

prompt = ChatPromptTemplate.from_messages([
    ("human", "{question}")
])

chain = prompt | model

response = chain.invoke({"question": "Tell me a joke."})

print(f"Model: {response.response_metadata.get("model_name")}")

result = parser.invoke(response)
print(f"Answer: {result}")




Model: claude-sonnet-4-5-20250929
Answer: Why don't scientists trust atoms?

Because they make up everything!
